In [74]:
from datasets import load_dataset
dataset = load_dataset("roneneldan/TinyStories")

print(dataset)
print(dataset['train'].features)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})
{'text': Value('string')}


In [75]:
small_train = dataset['train'].select(range(10_000))
small_eval = dataset['validation'].select(range(1_000))

all_text = ''.join(small_train['text']) + ''.join(small_eval['text'])

vocab = sorted(set(all_text))
print(vocab)
print(len(vocab))

print(small_train)

['\n', ' ', '!', '"', '$', '&', "'", '*', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '¡', '¦', '©', '«', '±', '³', '»', 'Â', 'Ã', 'â', 'œ', 'Š', '˜', '“', '”', '€', '™']
94
Dataset({
    features: ['text'],
    num_rows: 10000
})


In [76]:
stoi = {ch:i for i,ch in enumerate(vocab)}
itos = {i:ch for i,ch in enumerate(vocab)}


# Simple Bijection Encoding, BPE not needed for such a small model
encode = lambda seq: [stoi[c] for c in seq]
decode = lambda tok: ''.join([itos[i] for i in tok])

def tokenize(example):
    return {'ids': encode(example['text'])}

tokenized_train = small_train.map(tokenize, remove_columns=['text'])
tokenized_eval   = small_eval.map(tokenize, remove_columns=['text'])


import numpy as np

#Flatten to 1-D arrays for torch
train_ids = np.array([id for row in tokenized_train['ids'] for id in row], dtype=np.uint16)
eval_ids   = np.array([id for row in tokenized_eval['ids']   for id in row], dtype=np.uint16)

In [77]:
import torch

train = torch.tensor(train_ids,dtype=torch.long)
eval = torch.tensor(eval_ids,dtype=torch.long)
print(train.shape,train.dtype)
print(train[:100])


torch.Size([8674761]) torch.int64
tensor([39, 64, 55,  1, 54, 51, 75,  8,  1, 51,  1, 62, 59, 70, 70, 62, 55,  1,
        57, 59, 68, 62,  1, 64, 51, 63, 55, 54,  1, 36, 59, 62, 75,  1, 56, 65,
        71, 64, 54,  1, 51,  1, 64, 55, 55, 54, 62, 55,  1, 59, 64,  1, 58, 55,
        68,  1, 68, 65, 65, 63, 10,  1, 43, 58, 55,  1, 61, 64, 55, 73,  1, 59,
        70,  1, 73, 51, 69,  1, 54, 59, 56, 56, 59, 53, 71, 62, 70,  1, 70, 65,
         1, 66, 62, 51, 75,  1, 73, 59, 70, 58])


In [78]:
block_size = 8

x = train[:block_size+1]
y = train[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"Context: {context}, Target:{target}")
   
   

Context: tensor([39]), Target:64
Context: tensor([39, 64]), Target:55
Context: tensor([39, 64, 55]), Target:1
Context: tensor([39, 64, 55,  1]), Target:54
Context: tensor([39, 64, 55,  1, 54]), Target:51
Context: tensor([39, 64, 55,  1, 54, 51]), Target:75
Context: tensor([39, 64, 55,  1, 54, 51, 75]), Target:8
Context: tensor([39, 64, 55,  1, 54, 51, 75,  8]), Target:1


In [79]:
torch.manual_seed(1337)
batch_size = 4 #How many independent sequences do we porcess in parallel
block_size = 8 #Maximum amount of context for transformer

def get_batch(split):
    data = train if split == 'train' else eval

    ix = torch.randint(len(data) -block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    return x,y

xb, yb = get_batch('train')
print(xb.shape,xb)
print(yb.shape,yb)

torch.Size([4, 8]) tensor([[70, 58, 55,  1, 54, 65, 65, 68],
        [62, 54, 10,  1, 36, 71, 53, 75],
        [ 1, 63, 65, 63,  1, 53, 51, 63],
        [65,  1, 53, 68, 75, 10,  1, 43]])
torch.Size([4, 8]) tensor([[58, 55,  1, 54, 65, 65, 68, 10],
        [54, 10,  1, 36, 71, 53, 75,  1],
        [63, 65, 63,  1, 53, 51, 63, 55],
        [ 1, 53, 68, 75, 10,  1, 43, 58]])


In [ ]:
import torch.nn as nn
from torch.nn import functional as F

class Model(nn.Module):
    
    def __init__(self, vocab_size):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)
    
    def forward(self, idx, targets=None):

        logits = self.token_embedding_table(idx) # Logits(Batch = 4, Time = 8, Channels = len(vocab))

        if targets is None: 
            loss = None
            return logits,loss
        else:

            B,T,C = logits.shape

            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits,targets)

        return logits,loss

    def generate(self,idx,max_new_tokens):

        for _ in range(max_new_tokens):

            logits, _ = self(idx)

            logits = logits[:, -1, :]
            probs = F.softmax(logits,dim=-1)

            idx_next = torch.multinomial(probs,num_samples=1)

            idx = torch.cat((idx,idx_next),dim=1)

        return idx

m = Model(vocab_size=len(vocab))
out,loss = m(xb,yb)
print(out.shape,out)
print(loss)

idx = torch.zeros((1,1),dtype=torch.long)
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist())) 

torch.Size([32, 94]) tensor([[ 0.4550,  0.4445, -0.6998,  ..., -0.9117, -0.0458, -0.5051],
        [ 0.4357,  0.5604,  1.4756,  ...,  0.1482,  0.1482, -0.8726],
        [ 2.3530,  0.8707,  0.7649,  ..., -0.5477, -1.6349, -0.3141],
        ...,
        [-0.1848, -0.6850,  1.5450,  ...,  0.8209, -0.1027, -0.0572],
        [ 0.1451,  0.0996,  1.0828,  ...,  1.9471, -0.5025,  0.7531],
        [ 1.2046,  1.0895,  1.1504,  ..., -1.3593, -1.1304,  1.5097]],
       grad_fn=<ViewBackward0>)
tensor(5.2882, grad_fn=<NllLossBackward0>)


<>:49: SyntaxWarning: 'int' object is not subscriptable; perhaps you missed a comma?
<>:49: SyntaxWarning: 'int' object is not subscriptable; perhaps you missed a comma?
/tmp/ipykernel_15844/4025431677.py:49: SyntaxWarning: 'int' object is not subscriptable; perhaps you missed a comma?
  print(decode(m.generate(idx,max_new_tokens=100[0]).tolist()))


TypeError: 'int' object is not subscriptable